# Teplota Českých Budějovic a Hluboké z Landsatu 8: Planckův zákon v praxi

[![Otevřít v Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OpenSkyLab-space/notebooks/blob/main/blackbody_radiation_hluboka_cs.ipynb)

**OpenSkyLab · notebook k laboratoři** [Záření černého tělesa](https://openskylab.space/edu_viz_blackbody_radiation.html) · [English version](blackbody_radiation_hluboka_en.ipynb)

Laboratoř ukazuje Planckův zákon, Wienův posun, čtvrtou mocninu Stefanova–Boltzmannova zákona a vliv emisivity. Tady stejné vzorce použiješ na skutečný termální snímek Landsatu 8 z vlny veder 26. června 2019 a z digitálních čísel spočítáš teplotu povrchu Českých Budějovic a Hluboké.

**Cíl:** převést termální pásmo Landsatu 8 z DN na zář a jasovou teplotu, opravit ji o emisivitu, odhadnout chybu ze špatné emisivity a odrazu oblohy a ukázat, proč družice hledají malé požáry ve 3,9 µm.

**Data:** Landsat 8 OLI/TIRS, Collection 1 Level-1, scéna `LC08_L1TP_191026_20190626_20190705_01_T1` (26. 6. 2019, 9:51 UTC, oblačnost 0,2 %), pásma B4, B5 a B10 a soubor MTL s kalibračními konstantami z veřejného archivu Google Cloud. Čte se jen výřez 16 × 16 km mezi Českými Budějovicemi a Hlubokou nad Vltavou (asi 3 MB).

**Úkoly:** pět. Odpovědi piš do buněk *Tvoje odpověď* pod úkoly. Řádky označené `# ◀ parametr` můžeš měnit a buňku spustit znovu (Shift + Enter).

**Čas:** asi 45 min. Potřebuješ Google účet (Colab); nic se neinstaluje do počítače. Buňky spouštěj popořadě shora dolů.

## 1. Příprava

**Povinná buňka, spusť ji jako první.** Doinstaluje knihovnu rasterio, která čte rastrová data (GeoTIFF) přímo z internetu, a načte knihovny a fyzikální konstanty. Kód je sbalený; zobrazíš ho pomocí **Show code**.

In [ ]:
# @title Příprava (spusť jako první) {display-mode: "form"}
%pip install -q rasterio

import matplotlib.pyplot as plt
import numpy as np
import rasterio
import requests
from pyproj import Transformer
from rasterio.windows import from_bounds
from scipy.constants import c, h, k, sigma

**Povinná buňka, spusť ji hned po přípravě.** Vložení nastavení je nepovinné: když do pole `Z_VIZUALU` vložíš nastavení zkopírované z laboratoře, převezmou buňky níže stejné hodnoty parametrů a u každé to vypíšou. Když pole necháš prázdné (`{}`), použijí se výchozí hodnoty. Buňku spusť v obou případech, protože definuje funkci `z_viz`, kterou používají další buňky. Nic nestahuje. Kód je sbalený; zobrazíš ho pomocí **Show code**.

In [ ]:
# @title Nastavení z laboratoře (spusť; vložení je nepovinné) {display-mode: "form"}
# @markdown V laboratoři klikni na odkaz Notebook. Nastavení se zkopíruje; vlož ho sem místo `{}` (Ctrl+V) a buňku spusť.
Z_VIZUALU = {}  # @param {type:"raw"}
STEM = "blackbody_radiation"


def z_viz(name, default, mapping=None):
    """Hodnota parametru z laboratoře, pokud ji vložený kód obsahuje; jinak výchozí hodnota."""
    z = (
        Z_VIZUALU
        if isinstance(Z_VIZUALU, dict) and Z_VIZUALU.get("osl") == STEM
        else {}
    )
    if name not in z:
        return default
    v = z[name]
    if mapping is not None:
        if v not in mapping:
            print(
                f"{name}: hodnota {v!r} z laboratoře v tomto notebooku není, ponechávám {default!r}"
            )
            return default
        v = mapping[v]
    elif isinstance(default, bool):
        v = bool(v)
    elif isinstance(default, int):
        v = int(round(v))
    elif isinstance(default, float):
        v = float(v)
    print(f"{name} = {v!r}  (z laboratoře)")
    return v


if isinstance(Z_VIZUALU, dict) and Z_VIZUALU and Z_VIZUALU.get("osl") != STEM:
    print(
        f"Vložený kód je z jiné laboratoře ({Z_VIZUALU.get('osl')}), notebook ho nepoužije."
    )

## 2. Planckův zákon, Wien a čtvrtá mocnina

### Z teploty na spektrální zář: Planckův zákon

Planckův zákon popisuje, jak záření černého tělesa závisí na vlnové délce a teplotě:

$$
L_\lambda(\lambda, T) =
\frac{2 h c^2}{\lambda^5}
\frac{1}{\exp\!\left(\frac{h c}{\lambda k T}\right)-1}.
$$

| Symbol | Význam | Jednotka ve vzorci |
| --- | --- | --- |
| $\lambda$ | vlnová délka | $\mathrm{m}$ |
| $T$ | teplota černého tělesa | $\mathrm{K}$ |
| $h$ | Planckova konstanta | $\mathrm{J\,s}$ |
| $c$ | rychlost světla ve vakuu | $\mathrm{m\,s^{-1}}$ |
| $k$ | Boltzmannova konstanta | $\mathrm{J\,K^{-1}}$ |

Spektrální zář $L_\lambda$ v tomto vzorci má jednotku
$\mathrm{W\,m^{-2}\,sr^{-1}\,m^{-1}}$: je vyjádřena na metr vlnové délky.

**Jak vzorec souvisí s kódem:** `planck(lam_um, T)` přijímá vlnovou délku v µm
a teplotu v K. První `1e-6` převádí vlnovou délku z µm na m. Poslední `1e-6`
převádí vypočtenou zář na $\mathrm{W\,m^{-2}\,sr^{-1}\,\mu m^{-1}}$, tedy na µm
vlnové délky, které používáme v notebooku.

`np.expm1(x)` počítá $e^x-1$; pro malé hodnoty $x$ je numericky přesnější
než přímý zápis `np.exp(x) - 1`.

In [ ]:
def planck(lam_um, T):
    """Spektrální zář černého tělesa ve W·m⁻²·sr⁻¹·µm⁻¹ pro vlnovou délku v µm a teplotu v K."""
    lam = np.asarray(lam_um) * 1e-6
    return 2 * h * c**2 / lam**5 / np.expm1(h * c / (lam * k * T)) * 1e-6

### Ze spektrální záře na jasovou teplotu

**Jasová teplota** $T_b$ je teplota černého tělesa, které by na dané vlnové délce
mělo naměřenou spektrální zář. Vyjádřením teploty z Planckova zákona dostaneme:

$$
T_b(\lambda, L_\lambda) =
\frac{h c}{\lambda k}
\frac{1}{\ln\!\left(1+\frac{2 h c^2}{\lambda^5 L_\lambda}\right)}.
$$

Symboly a jednotky jsou stejné jako výše: $\lambda$ je v metrech a $L_\lambda$
je zář na metr vlnové délky. Výsledná teplota $T_b$ je v kelvinech.
U skutečného povrchu se jasová teplota může lišit od jeho teploty; vliv
emisivity prozkoumáme v dalších částech notebooku.

**Jak vzorec souvisí s kódem:** `planck_inv(lam_um, L)` přijímá vlnovou délku
v µm a zář v $\mathrm{W\,m^{-2}\,sr^{-1}\,\mu m^{-1}}$. Vlnovou délku proto
násobí `1e-6` a zář násobí `1e6`, aby ve vzorci byly jednotky na metr.
Funkce vrací jasovou teplotu v K.

`np.log1p(x)` počítá přirozený logaritmus $\ln(1+x)$; pro malé hodnoty $x$
je numericky přesnější než přímý zápis `np.log(1 + x)`.

In [ ]:
def planck_inv(lam_um, L):
    """Jasová teplota (K): teplota černého tělesa, které by mělo zář L na vlnové délce lam_um."""
    lam = lam_um * 1e-6
    return h * c / (lam * k) / np.log1p(2 * h * c**2 / lam**5 / (np.asarray(L) * 1e6))

> **💻 Co dělá kód:** Pro každou teplotu ze seznamu `T_list` vykreslí Planckovu křivku od 0,1 do 100 µm a vypíše vlnovou délku maxima (Wienův zákon), celkový tok spočítaný integrací křivky spolu s kontrolou σT⁴ a zář na 10,9 µm. Šedě je vyznačené termální okno 8–14 µm.

In [ ]:
# Krajina, kamna, požár, povrch Slunce.
T_list = [300, 600, 900, 5772]  # ◀ parametr: teploty v K
lam = np.logspace(-1, 2, 2000)  # 0,1 až 100 µm
fig, ax = plt.subplots(figsize=(9, 4))
for T in T_list:
    L = planck(lam, T)
    ax.loglog(lam, L, label=f"{T} K")
    lmax = 2897.77 / T  # Wienův posunovací zákon, µm
    M = np.trapezoid(L, lam) * np.pi  # zář → zářivý tok z plochy (W·m⁻²)
    print(
        f"{T:5d} K: maximum {lmax:6.2f} µm, tok {M:10.4g} W/m² (σT⁴ = {sigma * T**4:10.4g}), "
        f"zář v 10,9 µm {planck(10.9, T):8.3g} W·m⁻²·sr⁻¹·µm⁻¹"
    )
ax.axvspan(8, 14, color="tab:red", alpha=0.08, label="termální okno 8–14 µm")
ax.set(
    xlabel="vlnová délka (µm)",
    ylabel="spektrální zář (W·m⁻²·sr⁻¹·µm⁻¹)",
    ylim=(1e-4, 1e8),
)
ax.legend()
plt.show()

> **✏️ Úkol 1.** Kolikrát vzroste celkový tok a kolikrát zář v 10,9 µm, když teplotu zdvojnásobíš z 300 na 600 K? Proč se obě čísla liší? Spočítej také, o kolik procent víc vyzáří asfalt o 45 °C než tráva o 25 °C.

*Tvoje odpověď:* 

## 3. Skutečná scéna: z DN na jasovou teplotu

Pásmo B10 (10,6–11,2 µm) ukládá digitální čísla (DN). Soubor MTL obsahuje koeficienty lineárního převodu na spektrální zář a konstanty $K_1$, $K_2$ pro převod záře na jasovou teplotu:

$$
L_\lambda = M_L \cdot DN + A_L, \qquad T_b = \frac{K_2}{\ln\!\left(\frac{K_1}{L_\lambda} + 1\right)}.
$$

$M_L$ a $A_L$ jsou `RADIANCE_MULT_BAND_10` a `RADIANCE_ADD_BAND_10`, $L_\lambda$ vychází ve W·m⁻²·sr⁻¹·µm⁻¹ a $T_b$ v kelvinech. Druhý vzorec je Planckův zákon zjednodušený na jedno pásmo: $K_1$ a $K_2$ v sobě mají konstanty $h$, $c$, $k$ a efektivní vlnovou délku pásma.

> **💻 Co dělá kód:** Stáhne metadata (soubor MTL) skutečné scény Landsatu 8 a z termálního pásma B10 načte jen výřez kolem zvoleného středu. Digitální čísla převede podle koeficientů z MTL na zář a pak na jasovou teplotu a vypíše jejich rozsah. Nakonec výsledek porovná s vlastní funkcí `planck_inv`.

In [ ]:
SCENE = "LC08_L1TP_191026_20190626_20190705_01_T1"
BASE = f"https://storage.googleapis.com/gcp-public-data-landsat/LC08/01/191/026/{SCENE}/{SCENE}"
mtl = dict(
    line.strip().split(" = ")
    for line in requests.get(BASE + "_MTL.txt", timeout=60).text.splitlines()
    if " = " in line
)


def num(key):
    return float(mtl[key])


# Střed výřezu (°) a poloviční šířka (m).
lon, lat, half = 14.46, 49.01, 8000  # ◀ parametr


def read_band(b):
    with rasterio.Env(
        GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
        GDAL_HTTP_MAX_RETRY=5,
        GDAL_HTTP_RETRY_DELAY=2,
    ):
        with rasterio.open(f"/vsicurl/{BASE}_B{b}.TIF") as r:
            x, y = Transformer.from_crs(4326, r.crs, always_xy=True).transform(lon, lat)
            return r.read(
                1,
                window=from_bounds(x - half, y - half, x + half, y + half, r.transform),
            ).astype(float)


dn10 = read_band(10)
L10 = num("RADIANCE_MULT_BAND_10") * dn10 + num("RADIANCE_ADD_BAND_10")
BT = num("K2_CONSTANT_BAND_10") / np.log(num("K1_CONSTANT_BAND_10") / L10 + 1)
print(
    "datum a čas:",
    mtl["DATE_ACQUIRED"],
    mtl["SCENE_CENTER_TIME"].strip('"'),
    "| výška Slunce",
    mtl["SUN_ELEVATION"],
    "°",
)
print(
    f"DN {dn10.min():.0f}–{dn10.max():.0f} → zář {L10.min():.2f}–{L10.max():.2f} W·m⁻²·sr⁻¹·µm⁻¹ → "
    f"jasová teplota {BT.min() - 273.15:.1f} až {BT.max() - 273.15:.1f} °C"
)
print(
    f"kontrola Planckem pro 10,9 µm: {planck_inv(10.9, np.median(L10)) - 273.15:.1f} °C vs. K1/K2: {np.median(BT) - 273.15:.1f} °C"
)

> **💻 Co dělá kód:** Vykreslí mapu jasové teploty z předchozí buňky ve stupních Celsia. Nic nepočítá.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(BT - 273.15, cmap="inferno")
fig.colorbar(im, ax=ax, label="jasová teplota B10 (°C)")
ax.set_title(
    f"České Budějovice a Hluboká, {mtl['DATE_ACQUIRED']}, Landsat 8 B10\n(pixel 30 m, převzorkováno ze 100 m)"
)
ax.axis("off")
plt.show()

> **✏️ Úkol 2.** Najdi na mapě Vltavu, Malši a rybníky u Hluboké, lesy a zastavěné plochy s velkými halami na okraji Českých Budějovic. Jaké jasové teploty mají? Proč je rozdíl mezi vodou a střechami hal kolem poledne tak velký? Proč není jasová teplota totéž co teplota povrchu?

*Tvoje odpověď:* 

## 4. Emisivita

Reálné povrchy vyzařují méně než černé těleso: $L_\lambda = \varepsilon \, B_\lambda(T)$, kde $B_\lambda$ je Planckova funkce a $\varepsilon$ emisivita (0 až 1). Emisivitu odhadneme z NDVI zjednodušenou metodou prahů podle Sobrina a kol. (2004): holý povrch 0,97, plná vegetace 0,99, mezi tím podle podílu vegetace s opravou na vícenásobné odrazy mezi rostlinami a půdou. Vodě (NDVI < 0) dáme typickou hodnotu 0,991. Odraz oblohy a atmosféru zatím zanedbáme.

> **💻 Co dělá kód:** Ze stejné scény načte červené a blízké infračervené pásmo, spočítá odrazivost a NDVI a z NDVI odhadne emisivitu každého pixelu (prahy Sobrina a kol.). Pak spočítá teplotu povrchu LST a vypíše, o kolik se liší od jasové teploty. Vykreslí mapy NDVI a této opravy.

In [ ]:
sun = np.sin(np.radians(num("SUN_ELEVATION")))


def refl(b):
    return (
        num(f"REFLECTANCE_MULT_BAND_{b}") * read_band(b)
        + num(f"REFLECTANCE_ADD_BAND_{b}")
    ) / sun


red, nir = refl(4), refl(5)
ndvi = (nir - red) / (nir + red)

eps_soil, eps_veg, eps_water = 0.97, 0.99, 0.991  # ◀ parametr: emisivity
ndvi_s, ndvi_v = 0.2, 0.5
pv = np.clip((ndvi - ndvi_s) / (ndvi_v - ndvi_s), 0, 1) ** 2  # podíl vegetace v pixelu
# poslední člen: dutinový efekt (vícenásobné odrazy mezi rostlinami a půdou), tvarový
# faktor 0,55
eps = np.where(
    ndvi < 0,
    eps_water,
    np.where(
        ndvi < ndvi_s,
        eps_soil,
        eps_veg * pv + eps_soil * (1 - pv) + (1 - eps_soil) * (1 - pv) * 0.55 * eps_veg,
    ),
)
# Teplota tělesa, které by s emisivitou ε mělo naměřenou zář.
LST = planck_inv(10.9, L10 / eps)
d = LST - BT
print(
    f"emisivita {eps.min():.3f}–{eps.max():.3f}; oprava LST − BT: medián {np.median(d):.2f} K, rozsah {d.min():.2f} až {d.max():.2f} K"
)
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
a = ax[0].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.8)
fig.colorbar(a, ax=ax[0], label="NDVI")
b = ax[1].imshow(d, cmap="viridis")
fig.colorbar(b, ax=ax[1], label="LST − BT (K)")
for x in ax:
    x.axis("off")
plt.show()

> **💻 Co dělá kód:** Pro povrch o známé teplotě `T_true` spočítá, jakou jasovou teplotu by naměřil senzor při emisivitě 0,98 až 0,85, a vypíše chybu v kelvinech. Odraz okolí zde zanedbává.

In [ ]:
# o kolik se splete senzor, když povrch s naměřenou zář má jinou emisivitu, než
# předpokládáme
T_true = 273.15 + 40  # ◀ parametr: skutečná teplota povrchu v K
for eps_true in (0.98, 0.95, 0.92, 0.85):
    L = eps_true * planck(10.9, T_true)
    print(
        f"ε = {eps_true:.2f}: jasová teplota {planck_inv(10.9, L) - 273.15:5.1f} °C, chyba {planck_inv(10.9, L) - T_true:+5.2f} K"
    )

> **✏️ Úkol 3.** Kde je oprava o emisivitu největší a proč? Zhruba o kolik kelvinů se senzor splete na každou setinu emisivity? Změň `eps_soil` na 0,95 (beton, asfalt) a sleduj, jak se změní oprava v zastavěných plochách.

*Tvoje odpověď:* 

## 5. Odraz oblohy

Povrch s emisivitou ε odráží 1 − ε dlouhovlnného záření, které na něj svítí z oblohy. Jasná obloha je v 10 µm velmi studená, zatažená má zhruba teplotu základny oblačnosti. Senzor dostane

$$
L_\lambda = \varepsilon \, B_\lambda(T) + (1 - \varepsilon) \, B_\lambda(T_\mathrm{obloha}).
$$

> **💻 Co dělá kód:** Pro čtyři materiály s různou emisivitou spočítá jasovou teplotu, kterou uvidí termokamera, když povrch odráží oblohu o −40, −10, +10 a +27 °C. Teplotu povrchu nastavuje parametr `T_surf` v buňce níže (může ji převzít z nastavení laboratoře).

In [ ]:
T_surf = z_viz("T_surf", 273.15 + 27)  # ◀ parametr: teplota povrchu
for name, eps_m in [
    ("tráva", 0.98),
    ("beton", 0.92),
    ("zoxidovaný plech", 0.6),
    ("lesklý hliník", 0.05),
]:
    row = []
    # Jasná obloha … nízká oblačnost … obloha teplá jako povrch.
    for T_sky_c in (-40, -10, 10, 27):
        L = eps_m * planck(10.9, T_surf) + (1 - eps_m) * planck(10.9, 273.15 + T_sky_c)
        row.append(f"{planck_inv(10.9, L) - 273.15:6.1f}")
    print(
        f"{name:18s} ε={eps_m:.2f}: jasová teplota při obloze −40/−10/+10/+27 °C:",
        " ".join(row),
        "°C",
    )

> **✏️ Úkol 4.** Proč lesklý plech pod jasnou oblohou vypadá v termokameře mrazivě studený, i když má 27 °C? Proč při obloze stejně teplé jako povrch nehraje emisivita žádnou roli? Za jakého počasí bys proto plánoval termální snímkování střech?

*Tvoje odpověď:* 

## 6. Požár menší než pixel

Pixel je směs: podíl $p$ hoří při teplotě požáru, zbytek má teplotu pozadí, takže $L_\lambda = p \, B_\lambda(T_\mathrm{požár}) + (1 - p) \, B_\lambda(T_\mathrm{pozadí})$. Jako pozadí vezmeme medián skutečné jasové teploty z výřezu (České Budějovice a Hluboká) a porovnáme pásmo 3,9 µm (požárová pásma MODIS a VIIRS leží mezi 3,7 a 4,1 µm) s pásmem 10,9 µm.

> **💻 Co dělá kód:** Modeluje pixel, ve kterém hoří jen malý podíl plochy: zář je vážený průměr požáru o teplotě `T_fire` a pozadí (medián skutečné jasové teploty výřezu). Vykreslí, o kolik se pixel ohřeje ve 3,9 a 10,9 µm, a vypíše, při jak malém podílu je pixel teplejší o 10 K.

In [ ]:
T_fire = z_viz("T_fire", 900)  # ◀ parametr: teplota požáru v K
T_bg = float(np.median(BT))
p = np.logspace(-5, -1, 200)  # podíl požáru v pixelu
fig, ax = plt.subplots(figsize=(8, 3.8))
for lam_b in (3.9, 10.9):
    L = p * planck(lam_b, T_fire) + (1 - p) * planck(lam_b, T_bg)
    ax.semilogx(p * 100, planck_inv(lam_b, L) - T_bg, label=f"{lam_b} µm")
ax.axhline(10, color="gray", ls=":")
ax.set(
    xlabel="podíl požáru v pixelu (%)",
    ylabel="zvýšení jasové teploty (K)",
    title=f"pozadí {T_bg - 273.15:.1f} °C, požár {T_fire} K",
)
ax.legend()
plt.show()
for lam_b in (3.9, 10.9):
    L = p * planck(lam_b, T_fire) + (1 - p) * planck(lam_b, T_bg)
    p10 = p[np.argmax(planck_inv(lam_b, L) - T_bg > 10)]
    print(
        f"{lam_b} µm: o 10 K teplejší pixel už při podílu {p10 * 100:.3f} % = {p10 * 1e6:.0f} m² z pixelu 1 × 1 km"
    )

> **✏️ Úkol 5.** Při jakém podílu požáru se pixel ohřeje o 10 K ve 3,9 µm a při jakém v 10,9 µm? Jakou plochu to je v pixelu MODIS 1 × 1 km? Vysvětli rozdíl Wienovým zákonem. Proč algoritmy ve dne hlídají i odražené Slunce ve 3,9 µm?

*Tvoje odpověď:* 

> **✏️ Rozšíření (nepovinné).** Porovnej se spolužákem, který spustil notebook pro jiné místo a den: jaký rozsah jasových teplot vyšel jemu? Kolik z rozdílu může jít na vrub jiného dne a hodiny, jiné výšky Slunce a jiného složení krajiny?

*Tvoje odpověď:* 

---
### Zdroje dat a literatura

- Landsat 8 OLI/TIRS Collection 1 Level-1, U.S. Geological Survey; veřejný archiv Google Cloud (`gcp-public-data-landsat`). Data USGS jsou volná bez omezení užití (public domain), s uvedením zdroje.
- Převod DN na zář a jasovou teplotu: USGS (2019). *Landsat 8 (L8) Data Users Handbook*, verze 5.0, kap. 5.
- Emisivita z NDVI: Sobrino, J. A., Jiménez-Muñoz, J. C., Paolini, L. (2004). Land surface temperature retrieval from LANDSAT TM 5. *Remote Sensing of Environment* 90, 434–440. https://doi.org/10.1016/j.rse.2004.02.003
- Detekce požárů ve 3,9 a 11 µm: Giglio, L., Schroeder, W., Justice, C. O. (2016). The collection 6 MODIS active fire detection algorithm and fire products. *Remote Sensing of Environment* 178, 31–41. https://doi.org/10.1016/j.rse.2016.02.054

### Jak citovat

OpenSkyLab (2026). Teplota Českých Budějovic a Hluboké z Landsatu 8: Planckův zákon v praxi [notebook, výukový materiál], verze 1.0. https://github.com/OpenSkyLab-space/notebooks/blob/main/blackbody_radiation_hluboka_cs.ipynb. Licence CC BY 4.0.

Text a kód notebooku: OpenSkyLab, licence [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Data mají licence svých poskytovatelů (viz výše).

*Materiál připravil tým OpenSkyLab s pomocí generativní AI Claude (Anthropic), která pomáhala s návrhem kódu a textu. Za obsah odpovídají autoři.*